# Approach B runbook: transformers

Run from the `resume_parser_transformer/` folder. Each cell calls a script, so the outputs below record training curves and metrics. The scripts are the source of truth.

| Step | Time |
|---|---|
| Section classifier | ~1 min |
| NER fine-tuning (4 GB GPU) | ~12 min |
| Evaluations | ~5 min |

CPU-only NER training works but is much slower.

In [10]:
import os
os.environ["USE_TF"] = "0"              # TensorFlow + Keras 3 is installed; keep transformers from importing it
os.environ["OTEL_SDK_DISABLED"] = "true"  # no telemetry exporter to localhost:4318
import platform, sys, spacy, sklearn, torch
print("Python", platform.python_version(), "| spaCy", spacy.__version__, "| scikit-learn", sklearn.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
import transformers, sentence_transformers
print("transformers", transformers.__version__, "| sentence-transformers", sentence_transformers.__version__)

Python 3.10.12 | spaCy 3.8.16 | scikit-learn 1.7.2 | torch 2.5.0+cu124
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
transformers 4.53.2 | sentence-transformers 4.1.0


## 1. Section classifier: MiniLM sentence embeddings + Logistic Regression

Same data, aliases and splits as Approach A's TF-IDF model. Re-run it whenever the alias list in `preprocessing/sections.py` changes.

In [11]:
!python3 training/train_section_classifier.py

train blocks=369 test blocks=140 labels={'PERSONAL_INFORMATION': 32, 'SKILLS': 54, 'OTHER': 17, 'EDUCATION': 31, 'PROJECTS': 47, 'SUMMARY': 24, 'EXPERIENCE': 132, 'CERTIFICATIONS': 31, 'VOLUNTEERING': 1}
fold 1: {'accuracy': 0.5246, 'macro_f1': 0.4712, 'weighted_f1': 0.5091}
fold 2: {'accuracy': 0.4273, 'macro_f1': 0.3084, 'weighted_f1': 0.4048}
fold 3: {'accuracy': 0.4964, 'macro_f1': 0.3539, 'weighted_f1': 0.4926}
fold 4: {'accuracy': 0.6159, 'macro_f1': 0.5233, 'weighted_f1': 0.6273}
                      precision    recall  f1-score   support

        ACHIEVEMENTS       0.00      0.00      0.00         2
      CERTIFICATIONS       0.42      0.83      0.56         6
           EDUCATION       0.75      0.55      0.63        11
          EXPERIENCE       0.43      0.60      0.50        47
     EXTRACURRICULAR       0.00      0.00      0.00         0
               OTHER       1.00      0.20      0.33         5
PERSONAL_INFORMATION       0.80      0.89      0.84         9
           

## 2. NER: fine-tune `roberta-base` (BIO token classification)

Uses the same splits as Approach A, with overlapping 512-token windows. The best epoch is picked by validation exact-span F1 on full resumes, with early stopping after 3 epochs. The test split is not used. This overwrites `models/resume_transformer/`, the model the parser loads.

In [12]:
!python3 training/train_transformer_ner.py --model-name roberta-base --epochs 15 --patience 3

device=cuda train resumes=248 windows=953 validation resumes=31 labels=['COMPANY', 'DATE', 'DEGREE', 'EMAIL', 'INSTITUTION', 'JOB_TITLE', 'LOCATION', 'PERSON', 'SKILL'] nested-span overlaps resolved=0
Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Could not load library libnvrtc.so.11.2. Error: libnvrtc.so.11.2: cannot open shared object file: No such file or directory
Could not load library libnvrtc.so.11.2. Error: libnvrtc.so.11.2: cannot open shared object file: No such file or directory
/home/shaanal/.local/lib/python3.10/site-packages/torch/autograd/graph.py:825: UserWarning: cuDNN SDPA backward got grad_output.strides() != output.strides(), attempting to materialize a grad_output with matching strides... (Triggered internally at ../aten/src/AT

## Evaluate on the held-out test split

Run once, after training. **Entity level:** precision, recall and F1 per label, with exact and overlap matching. **Section level:** per-line accuracy, comparing rules only, classifier only and the hybrid. **Field and document level:** baseline vs full system. Each script also saves JSON to `evaluation/results/`.

In [1]:
!python3 evaluation/evaluate_ner.py

Approach B: fine-tuned RoBERTa NER + sentence-embedding sections + embedding skill matching
Evaluated resumes: 31; mean NER time per resume: 928 ms
LABEL           EXACT P      R     F1   OVERLAP P      R     F1  SUPPORT
COMPANY           0.570  0.652  0.608       0.703  0.804  0.750      112
DATE              0.759  0.797  0.777       0.853  0.896  0.874      182
DEGREE            0.476  0.591  0.527       0.658  0.818  0.730       66
EMAIL             0.964  1.000  0.982       0.964  1.000  0.982       54
INSTITUTION       0.489  0.613  0.544       0.660  0.827  0.734       75
JOB_TITLE         0.588  0.677  0.629       0.756  0.870  0.809      192
LOCATION          0.706  0.706  0.706       0.814  0.814  0.814      177
PERSON            0.881  0.897  0.889       0.898  0.914  0.906       58
SKILL             0.515  0.571  0.542       0.624  0.693  0.657      576
----------------------------------------------------------------------
MICRO             0.603  0.665  0.633       0.720  

In [2]:
!python3 evaluation/evaluate_sections.py

Approach B: fine-tuned RoBERTa NER + sentence-embedding sections + embedding skill matching
Held-out resumes: 3  lines: 202
SYSTEM              ACCURACY  MACRO F1  WEIGHTED F1
rules only             0.579     0.416        0.530
classifier only        0.455     0.367        0.438
hybrid                 0.639     0.505        0.587

Hybrid per-section report:
                      precision    recall  f1-score   support

        ACHIEVEMENTS       0.00      0.00      0.00         2
      CERTIFICATIONS       1.00      0.67      0.80         6
           EDUCATION       0.44      1.00      0.61        11
          EXPERIENCE       0.56      0.97      0.71        63
               OTHER       0.20      0.09      0.12        11
PERSONAL_INFORMATION       0.71      0.86      0.77        14
            PROJECTS       1.00      0.82      0.90        34
              SKILLS       0.57      0.14      0.22        29
             SUMMARY       1.00      0.25      0.40        32

            accura

Saved /home/shaanal/Documents/EHS/Project_Resume_Parser/resume_parser_transformer/evaluation/results/sections.json


In [3]:
!python3 evaluation/evaluate_fields.py

Approach B: fine-tuned RoBERTa NER + sentence-embedding sections + embedding skill matching
Resumes: 31
FIELD            baseline (rules + ontology)                   full system
full_name                              0.857                         0.964
email                                  1.000                         1.000
location                               0.567                         0.767
skills                                 0.381                         0.559
degree                                 0.603                         0.753
institution                            0.622                         0.643
company                                0.291                         0.656
designation                            0.373                         0.667
DOC MEAN                               0.596                         0.755
DOC CORRECT                            0.032                         0.032
Saved /home/shaanal/Documents/EHS/Project_Resume_Parser/resume_parser_t

## Summary of this run

In [4]:
import json
from pathlib import Path
r = lambda name: json.loads(Path("evaluation/results", name).read_text())
ner, sec, fields = r("ner_test.json"), r("sections.json"), r("fields_test.json")
print(ner["approach"])
print(f"Entity micro F1 (31 test resumes): exact {ner['micro']['exact']['f']:.3f} | overlap {ner['micro']['overlap']['f']:.3f} | {ner['ms_per_resume']:.0f} ms/resume")
for row in sec["results"]:
    print(f"Sections, {row['system']:<16} accuracy {row['accuracy']:.3f}  macro-F1 {row['macro_f1']:.3f}")
base, full = fields["results"]["baseline (rules + ontology)"], fields["results"]["full system"]
print(f"\n{'FIELD':<14}{'baseline':>10}{'full':>8}")
for name in full["fields"]:
    b, f = base["fields"][name]["score"], full["fields"][name]["score"]
    print(f"{name:<14}{b:>10.3f}{f:>8.3f}")
print(f"{'DOC MEAN':<14}{base['document_mean_score']:>10.3f}{full['document_mean_score']:>8.3f}")

B: fine-tuned RoBERTa NER + sentence-embedding sections + embedding skill matching
Entity micro F1 (31 test resumes): exact 0.633 | overlap 0.756 | 928 ms/resume
Sections, rules only       accuracy 0.579  macro-F1 0.416
Sections, classifier only  accuracy 0.455  macro-F1 0.367
Sections, hybrid           accuracy 0.639  macro-F1 0.505

FIELD           baseline    full
full_name          0.857   0.964
email              1.000   1.000
location           0.567   0.767
skills             0.381   0.559
degree             0.603   0.753
institution        0.622   0.643
company            0.291   0.656
designation        0.373   0.667
DOC MEAN           0.596   0.755


## Parse the sample resumes and refresh the sample outputs

Batch output in `data/parsed/` uses the lean schema; add `--with-metadata` to keep confidence, evidence and the detected sections.

In [5]:
!python3 batch_parse.py

Components: {'approach': 'B: fine-tuned RoBERTa NER + sentence-embedding sections + embedding skill matching', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'ner_model': 'models/resume_transformer', 'ner_base_model': 'roberta-base', 'section_model': 'models/section_classifier/section_head.joblib'}


[SUCCESS] 1.txt


[SUCCESS] 10.docx


[SUCCESS] 14.docx


[SUCCESS] 15.docx


...

[SUCCESS] CV_RashidSaid.pdf


[SUCCESS] CV_Robert Cooper.jpg


[SUCCESS] CV_ShihabAmri.pdf


[SUCCESS] CV_SoukeurAbdelhamid.jpg


[SUCCESS] CV_Yassine.pdf
Completed 33/33


In [6]:
!python3 -m inference.cli data/input/CV_GhandiAli.pdf --output sample_output.json > /dev/null
!python3 -m inference.cli data/input/CV_GhandiAli.pdf --with-metadata --output sample_output_with_metadata.json > /dev/null
!python3 schema.py sample_output.json sample_output_with_metadata.json

sample_output.json: OK
sample_output_with_metadata.json: OK


## Web demo and API

Run this in a terminal, not here, because the server keeps running: `uvicorn api.app:app --reload` (add `--port 8001` if the other project's server is on 8000).

Open http://127.0.0.1:8000, choose a resume and click **Parse**. The result appears as readable sections, with **Show JSON** / **Download JSON** for the raw output. The JSON API is documented at http://127.0.0.1:8000/docs (`POST /parse-resume`, optional `include_metadata=true`).